# Steam Sales Dataset Analysis (LLM Batch Preprocessing)

This project focuses on LLM-based batch preprocessing to clean, standardize, and enrich large-scale Steam marketplace data before analysis. The goal is to transform raw, inconsistent data into a structured and analysis-ready format using Large Language Models (LLMs), enabling more accurate and meaningful downstream insights.

Batch processing is a data processing approach where large volumes of data are handled in groups (batches) rather than processing each record individually in real-time.

It is especially useful for large datasets where:

- Real-time processing is unnecessary
- Efficiency and scalability are priorities
- Data can be processed asynchronously

LLM Batch Preprocessing Workflow:

- Data Chunking - The dataset is split into smaller, manageable batches.
- LLM Invocation - Send each batch to the LLM with clear prompt instructions for preprocessing.
- Data Cleaning - Fix missing values, remove noise, and standardize formats.
- Data Enrichment -Convert unstructured data into structured form and add useful features.
- Validation & Merging - Check consistency and combine all processed batches into the final dataset.

## Import Libraries

In [ ]:
import os
import requests 
import subprocess 
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
from openai import OpenAI
from litellm import completion
from pathlib import Path
import random
import json
from sales_util import item_parser, item_preprocess
from sales_util.items_data import Item

In [ ]:
# Load env file
load_dotenv(override=True)

In [ ]:
BASE_DIR = Path(os.getenv("PROJECT_ROOT"))

## Load Dataset From HuggingFace

In [ ]:
LITE_MODE = True

In [ ]:
username = "KumudithaSilva"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

In [ ]:
for index, item in enumerate(items):
    item.id = index

In [ ]:
items[0].id

## Ollama Initialization

In [ ]:
subprocess.Popen("ollama serve", shell=True)

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
OLLAMA_API_URL = "http://localhost:11434/v1"

## LLM Batch Processing Prompt

In [ ]:
SYSTEM_PROMPT = """Write a concise game description in 5–10 words.

Respond strictly in this format:
description: <short description>
"""

In [ ]:
sample_item = items[2].small_description
sample_item

In [ ]:
MODEL_OLLAMA = "ollama/llama3.2"
MODEL_OPEN_AI = "openai/gpt-4o-mini"

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sample_item}]
response = completion(messages=messages, model=MODEL_OLLAMA)
# response = completion(messages=messages, model=MODEL_OPEN_AI)

In [ ]:
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

##  JSONL Files

In [ ]:
def make_jsonl(item):
    body = {"model": "gpt-4o-mini", "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.small_description}]}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
make_jsonl(items[0])

In [ ]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
JSONL_PATH = BASE_DIR / "data"

In [ ]:
make_file(0, 22000, f"{JSONL_PATH}/lite.jsonl")

## Batch Mode

In [ ]:
openai = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

#### Uploads JSONL input file to OpenAI servers

In [ ]:
with open(f"{JSONL_PATH}/lite.jsonl", "rb") as f:
    response = openai.files.create(file=f, purpose="batch")
response

In [ ]:
file_id = response.id
file_id

### Run uploaded file through the model in bulk

In [ ]:
response = openai.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
# batch_69e3a5d82a488190bf3c1a9a4b79047f
result = openai.batches.retrieve(response.id)
result

In [ ]:
if result.status == "completed":
    response = openai.files.content(result.output_file_id)
    response.write_to_file(f"{JSONL_PATH}/batch_results.jsonl")
else:
    print("Batch Data in progress")

## Updated Dataset Description Section

In [ ]:
with open(f"{JSONL_PATH}/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])

        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        description = summary.split(":")[-1].strip()
        
        items[id].small_description = description

## Push To HuggingFace

In [ ]:
username = "KumudithaSilva"
lite = f"{username}/items_llm_raw_lite"

In [ ]:
random.seed(42)

shuffle_items = items.copy()
random.shuffle(shuffle_items)

In [ ]:
train = shuffle_items[:17_600]
val = shuffle_items[17_600:19_800]
test = shuffle_items[19_800:]

Item.push_to_hub(lite, train, val, test)